# NewsAPI

## Set Up

In [9]:
import os
from dotenv import load_dotenv, dotenv_values
from pathlib import Path

In [10]:
load_dotenv(Path("/Users/tanaphan/Documents/STUDY/UoBristol/Final_Project/team_21_lloyds/.env"))

True

In [11]:
API_KEY_NEWS = os.getenv("API_KEY_NEWS")

print(API_KEY_NEWS)

5f1946f04b7642ab944af4806cdc7e4b


## EDA

In [12]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#https://newsapi.org/v2/everything
#https://newsapi.org/v2/top-headlines

# FUNCTION: Pull NewsAPI - Everything Endpoint
def pull_news(KEYWORD):
    response = requests.get("https://newsapi.org/v2/everything", params={
        "q": f'"{KEYWORD}"', # Search using this Keyword
        "language": "en",
        "sortBy": "publishedAt", # relevancy, popularity, publishedAt
        "pageSize": 100, # 100 Max
        "page": 1,
        "apiKey": API_KEY_NEWS,
    })
    data = response.json()
    articles = data.get("articles", [])
    print(f"API status     : {data.get('status')}")
    print(f"Total results: {data.get('totalResults')}")
    print(f"Total articles pulled: {len(articles)}")
    if data.get("status") == "error":
        print(f"Error code   : {data.get('code')}")
        print(f"Error message: {data.get('message')}")
    df = pd.json_normalize(articles)
    return df

In [13]:
# Set Keyword
KEYWORD = "tesco"

print("="*60)
print(f"News for {KEYWORD}")
print("="*60)

df = pull_news(KEYWORD)

# Shape
print(f"\nShape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumns and types:")
print(df.dtypes)

News for tesco
API status     : ok
Total results: 205
Total articles pulled: 74

Shape: 74 rows x 9 columns

Columns and types:
author         object
title          object
description    object
url            object
urlToImage     object
publishedAt    object
content        object
source.id      object
source.name    object
dtype: object


In [14]:
print(df.head)

<bound method NDFrame.head of             author                                              title  \
0          Reuters  Deadly ‘Omega’ heat wave cooking Europe expect...   
1      Andrew Levy  Career criminal jailed for third time in 18 mo...   
2        ET Online  Noel Tata exiting Trent: The quiet Tata who bu...   
3     Caitlin Leng  Urgent recall on apples and kiwi fruit sold at...   
4       Aengus Cox  Late payments among issues for suppliers, surv...   
..             ...                                                ...   
69    Dirk Petzold  DASH Water Pink Lady Apple Packaging by Horse ...   
70  Helena Nicklin  I'm a wine expert: Here's how you can cut your...   
71    Ethan Ennals  How following a trendy high-protein diet could...   
72  Will Hallowell  Council enforcement officers are sacked after ...   
73            None  He runs instead of flies but Singapore's very ...   

                                          description  \
0   The record-breaking heat caused 

In [15]:
# Missing Values
print("Missing Value:")
print(df.isnull().sum())

# Duplication
print("\nDuplicatation:")
dup_title = df.duplicated(subset=["title"]).sum()
dup_description = df.duplicated(subset=["description"]).sum()
dup_url   = df.duplicated(subset=["url"]).sum()

print(f"Duplicate titles : {dup_title}")
print(f"Duplicate description : {dup_description}")
print(f"Duplicate URLs   : {dup_url}")

Missing Value:
author          7
title           0
description     2
url             0
urlToImage      7
publishedAt     0
content         0
source.id      59
source.name     0
dtype: int64

Duplicatation:
Duplicate titles : 5
Duplicate description : 3
Duplicate URLs   : 0


In [16]:
# Articles overtime
print("Article Timeline")
df["publishedAt"] = pd.to_datetime(df["publishedAt"])
df["date"] = df["publishedAt"].dt.date # create a column "date"
daily_counts = df.groupby("date").size().reset_index(name="count")
print(daily_counts)
plt.figure(figsize=(len(daily_counts) * 0.7, 4))
plt.plot(daily_counts["date"], daily_counts["count"], marker="o", color="blue")
plt.fill_between(daily_counts["date"], daily_counts["count"], alpha=0.1, color="blue")
plt.title(f"Articles per day | Keyword = {KEYWORD}")
plt.xlabel("Date")
plt.ylabel("Article Count")
plt.savefig(f"newsapi_{KEYWORD}_timeline.png") # Save graph
plt.close()

# Source Distribution
print("\nSource Distribution")
source_counts = df.groupby("source.name").size().reset_index(name="count").sort_values("count", ascending=False)
print(source_counts)
plt.figure(figsize=(10, len(source_counts) * 0.4))
plt.barh(source_counts["source.name"], source_counts["count"], color="green")
plt.title(f"Articles by source | Keyword = {KEYWORD}")
plt.xlabel("Article count")
plt.savefig(f"newsapi_{KEYWORD}_sources.png", bbox_inches="tight")
plt.close()

# Text Length
df["title_len"] = df["title"].fillna("").apply(len)
df["description_len"] = df["description"].fillna("").apply(len)

# Save as .CSV file
output_cols = ["date", "source.name", "title", "description", "content",
               "title_len", "description_len"]
df[output_cols].to_csv(f"newsapi_eda_{KEYWORD}.csv", index=False)

Article Timeline
          date  count
0   2026-06-13      2
1   2026-06-14      2
2   2026-06-15      4
3   2026-06-16      1
4   2026-06-17     13
5   2026-06-18     18
6   2026-06-19     11
7   2026-06-20      6
8   2026-06-22      4
9   2026-06-23      9
10  2026-06-24      4

Source Distribution
                   source.name  count
6                Dailymail.com     20
20                Slashdot.org      6
18                         RTE      6
22             The Irish Times      5
2                   Biztoc.com      4
16          New Zealand Herald      3
25             Theregister.com      3
29         Yahoo Entertainment      2
4           ComputerWeekly.com      2
24          The Times of India      2
19       Regionalmedianews.com      1
21                   TechRadar      1
0                  4sysops.com      1
23                The Next Web      1
27           Weandthecolor.com      1
28                  Www.gov.uk      1
26           Tom's Hardware UK      1
15            